# Practice 44 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — employee records

Use the DataFrame below.

- Extract `emp_num` (e.g. `'045'`) and `state` (e.g. `'CA'`) from `employee_id`.
- Add `is_internal`: True where `email` ends in `corp.com`.
- What is the mean salary for internal vs external employees? One groupby.
- Add `salary_band`: `'junior'` (< 75k), `'mid'` (75k–89k), `'senior'` (≥ 90k). Vectorized — no `.apply()`.
- Use a list comprehension to collect the names of all columns with `object` dtype.

In [60]:
employees = pd.DataFrame({
    'employee_id': ['EMP-001-NY', 'EMP-045-CA', 'EMP-012-TX', 'EMP-089-NY', 'EMP-003-CA', 'EMP-027-TX'],
    'email':       ['alice@corp.com', 'bob@partner.org', 'carol@corp.com', 'dave@external.net', 'eve@corp.com', 'frank@partner.org'],
    'salary':      [72000, 85000, 91000, 67000, 78000, 95000],
    'department':  ['Engineering', 'Sales', 'Engineering', 'Marketing', 'Sales', 'Engineering'],
})

# Your code here

employees['emp_num'] = employees['employee_id'].str.extract(r'(\d+)')
employees['state'] = employees['employee_id'].str.extract(r'(?:-)([A-Z]+)')
employees['is_internal'] = employees['email'].str.contains('corp.com$')
employees.groupby('is_internal')['salary'].mean()
employees['salary_band'] = pd.cut(employees['salary'],bins=[0,75000,90000,np.inf],
                                  labels=['junior','mid','senior'],
                                  right=False)

[c for c in employees.columns if employees[c].dtype == 'object']

['employee_id', 'email', 'department', 'emp_num', 'state']

---
## Level 2 — log parsing

Each row in `logs` is a raw server log string. Parse it into separate columns — figure out the regex yourself — then answer these questions:

1. Which user triggered the most errors?
2. What is the most common status code overall?
3. Among `ERROR` rows, which endpoint appeared most often?
4. What fraction of this user's requests resulted in an error?

In [58]:
logs = pd.DataFrame({'log': [
    '[2024-01] ERROR user_id=1042 endpoint=/api/search  status=500',
    '[2024-01] INFO  user_id=0821 endpoint=/api/login   status=200',
    '[2024-02] ERROR user_id=1042 endpoint=/api/cart    status=503',
    '[2024-02] INFO  user_id=0334 endpoint=/api/search  status=200',
    '[2024-03] ERROR user_id=0821 endpoint=/api/login   status=401',
    '[2024-03] INFO  user_id=1042 endpoint=/api/profile status=200',
    '[2024-03] ERROR user_id=0334 endpoint=/api/cart    status=500',
    '[2024-04] INFO  user_id=0821 endpoint=/api/search  status=200',
    '[2024-04] ERROR user_id=1042 endpoint=/api/login   status=401',
    '[2024-04] INFO  user_id=0334 endpoint=/api/profile status=200',
]})

# Your code here
logs['user_id'] = logs['log'].str.extract(r'user_id=(\d+)')
logs['type'] = logs['log'].str.extract(r'] ([A-Z]+)')
logs['code'] = logs['log'].str.extract(r'status=(\d+)')
logs['endpoint'] = logs['log'].str.extract(r'endpoint=(\S+)')

#1
er = logs[logs['type']=='ERROR']
u = er['user_id'].value_counts().idxmax()

#2
logs['code'].value_counts().idxmax()

#3 
er['endpoint'].value_counts().idxmax()

#4

np.mean(logs[logs['user_id'] == u]['type'] == 'ERROR')




np.float64(0.75)

In [59]:
logs['log'].str.extract(r'] ([A-Z]+)')

,0
0,ERROR
1,INFO
2,ERROR
3,INFO
4,ERROR
5,INFO
6,ERROR
7,INFO
8,ERROR
9,INFO


---
## Level 3 — SKU pipeline

Write a `.pipe()` chain with two functions:

- **`parse(df)`** — extract `category` (letters before the first `-`), `item_num` (digits after the last `-`), and convert `price` from `'$X.XX'` to float.
- **`enrich(df)`** — add `margin` = `(price - cost) / price`. Add `price_tier` with `np.select()`: `'budget'` (< 50), `'mid'` (50–499), `'premium'` (≥ 500). Add `is_high_margin`: True where `margin > 0.4`.

After the chain:
- Which `category` has the highest mean `margin`?
- What fraction of total `units_sold` came from `'premium'` tier items?
- Build `{category: total_units_sold}` using a dict comprehension from a groupby result.

In [50]:
products = pd.DataFrame({
    'sku':        ['ELEC-TV-055', 'CLTH-SHRT-012', 'ELEC-PHN-099', 'FURN-SOFA-003', 'CLTH-SHRT-041', 'FURN-DESK-017', 'ELEC-TV-082'],
    'price':      ['$499.99', '$29.99', '$799.00', '$1299.00', '$34.99', '$449.00', '$649.99'],
    'cost':       [310.0, 15.0, 420.0, 700.0, 12.0, 200.0, 390.0],
    'units_sold': [42, 130, 67, 18, 210, 35, 55],
})

# Your code here
def parse(df):
    df['category'] = df['sku'].str.extract(r'([A-Z]+)(?:-)')
    df['item_num'] = df['sku'].str.extract(r'(?:-)(\d+)$')
    df['price'] = df['price'].str.extract(r'([^$]+)').astype(float)
    return df 

def enrich(df):
    df['margin'] = (df['price'] - df['cost'])/df['price']
    conds =[df['price']< 50, df['price']<500]
    chos = ['buget','mid']
    df['price_tier'] = np.select(conds, chos, default='premium')
    df['is_high_margin'] = df['margin']>0.4

    return df 

res = products.copy().pipe(parse).pipe(enrich)

res.groupby('category')['margin'].mean().idxmax()

tot = res.groupby('price_tier')['units_sold'].sum()

tot['premium']/tot.sum()
ct = res.groupby('category')['units_sold'].sum()

{c:t for c, t in zip(ct.index, ct)}

{'CLTH': 340, 'ELEC': 164, 'FURN': 53}